## Process data 
将DB_wrapped.xml(包装好的梦境文本数据) 过一遍大模型，提取出需要的实体数据

In [1]:
# from utils folder import the function we need
import sys
import os

sys.path.append(os.path.abspath("../utils"))

from CallLLM import call_llm
from SetupLogging import setup_logging

In [2]:
# set up a logger
logger = setup_logging("process_data")

In [3]:
system_prompt = """
# Role
You are a senior expert in Computational Linguistics, Affective Computing, and Oneirology (Dream Science), specializing in extracting structured entity relations, behavioral sequences, and emotional polarity from large-scale unstructured dream narratives.
"""

user_prompt = """
# Background
I am conducting a high-throughput study on the topological and emotional features of dream structures. This research requires analyzing multiple dreams simultaneously to calculate entity frequency (Zipf's Law), character interaction networks, sequence mapping (for directed graph networks), and affective distribution (positive vs. nightmare states).

# Input Data Format
You will receive a batch of multiple dream texts. Each dream is wrapped in explicit XML tags with a unique identifier, like this:
<dream id="alta_1"> [Dream text content here] </dream>
<dream id="alta_2"> [Dream text content here] </dream>

# Task
Analyze each dream text in the provided batch independently and execute the following five tasks for each entry:
1. **Named Entity Recognition (NER)**: Extract all significant entities. Categories include: Characters (e.g., "mother", "stranger"), Objects (e.g., "wolf", "key"), Locations (e.g., "forest"), and Natural Phenomena (e.g., "rain").
2. **Global Entity Frequency**: Count the exact total occurrences of *every* unique entity (including characters) within that specific dream.
3. **Character Frequency Isolation**: Isolate entities categorized strictly as **Characters** (human, human-like figures, or personified entities, e.g., "mother", "ghost", "speaking dog") and count their frequencies independently.
4. **Textual Sequence Extraction**: List all entities in the exact order they appear in the reading flow. Record recurring entities multiple times to capture "looping" or "jumping" characteristics.
5. **Sentiment Classification**: Evaluate the overall emotional tone of the dream. Classify it as a boolean value: `true` if it is a positive/neutral-pleasant dream, and `false` if it is a negative dream/nightmare (characterized by fear, anxiety, pursuit, or distress).

# Constraints & Rules (Critical)
1. **Entity Atomization & Lemmatization**: Extract only the core noun in **lowercase** and **singular form** (e.g., "a massive black wolf" -> "wolf", "my mother's old houses" -> "house").
2. **Pronoun Resolution**: Map pronouns ("he", "it", "the beast") back to their specific antecedent entity if clearly identifiable.
3. **Sequence Order**: Follow the strict **Textual Reading Order** from first word to last, not the chronological plot backstory.
4. **Character Definition**: A "Character" is defined as any entity capable of agency, speech, or intentional behavior within the dream context.
5. **Data Consistency**: Ensure that every key in `character_frequency` is also present in `entities_frequency` with the exact same frequency count.
6. **Independent Evaluation**: Do not let the sentiment or entities of one dream cross-contaminate another dream in the batch.
7. **Language**: The entire JSON output (including summaries and entity names) must be in **English**.

# Output Format
Respond ONLY with a valid JSON array containing objects for each dream. Do not include any markdown conversational text outside the code block.

[
  {
    "dream_id": "alta_1",
    "dream_summary": "A concise one-sentence summary of the dream theme.",
    "positive_or_not": true,
    "entities_frequency": {
      "mother": 3,
      "wolf": 2,
      "forest": 1
    },
    "character_frequency": {
      "mother": 3
    },
    "entity_sequence": ["mother", "forest", "wolf", "mother", "wolf", "mother"],
    "metadata": {
      "total_unique_entities": 3,
      "total_unique_characters": 1,
      "sequence_total_length": 6
    }
  }
]

Input Data:
"""

In [4]:
# read in the wrapped data
data_file_path = "../../Datasets/DB_wrapped.xml"
data = None
with open(data_file_path, "r", encoding="utf-8") as f:
    data = f.readlines()

## 具体处理数据 - 批量处理逻辑

In [ ]:
BATCH_SIZE = 5  # 5 dreams per LLM call
SAVE_INTERVAL = 20  # Save every 20 batches (100 dreams)
MAX_CONCURRENCY = 10
OUTPUT_FILE = "../../Datasets/DB_LLM_processed.jsonl"
FAILED_FILE = "../../Datasets/DB_failed_batches.json"

import re
import json
import asyncio
import aiohttp
from tqdm import tqdm

# Extract dream_id from line
def extract_dream_id(line):
    match = re.search(r'<dream id="([^"]+)">', line)
    return match.group(1) if match else None

In [ ]:
#Read already processed dream_ids for resumability
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                processed_ids.add(item["dream_id"])
            except json.JSONDecodeError:
                continue
    logger.info(f"Found {len(processed_ids)} already processed dreams")

# Build list of (dream_id, dream_text) to process
dreams_to_process = []
for line in data:
    dream_id = extract_dream_id(line)
    if dream_id and dream_id not in processed_ids:
        dreams_to_process.append((dream_id, line.strip()))

logger.info(f"Total dreams to process: {len(dreams_to_process)}")

2026-05-19 00:53:41,782 - INFO - Found 3100 already processed dreams
2026-05-19 00:53:41,816 - INFO - Total dreams to process: 42121


In [ ]:
# Helper Functions to check the result
def is_error_result(result):
    """Check if the LLM result is an error message"""
    if result is None:
        return True
    return result.startswith(("Error:", "Exception:", "Failed")) or result.strip() == ""

def validate_result(result_list, expected_ids):
    """Validate LLM output - check dream_ids match and required fields exist"""
    if not isinstance(result_list, list):
        return False, "Result is not a list"
    
    result_ids = {item.get("dream_id") for item in result_list}
    if result_ids != set(expected_ids):
        return False, f"ID mismatch: expected {expected_ids}, got {result_ids}"
    
    required_fields = ["dream_id", "dream_summary", "positive_or_not", 
                       "entities_frequency", "character_frequency", "entity_sequence"]
    for item in result_list:
        for field in required_fields:
            if field not in item:
                return False, f"Missing field '{field}' in {item.get('dream_id')}"
    
    return True, "Valid"

def extract_json_from_response(result):
    """Extract JSON array from LLM response that may contain markdown or extra text"""
    json_match = re.search(r'\[\s*\{.*\}\s*\]', result, re.DOTALL)
    if json_match:
        return json_match.group(0)
    return result

In [ ]:
# Async Batch Processing Function
async def process_batch(batch_items, session, semaphore):
    """
    Process a batch of dreams via LLM API
    batch_items: list of (dream_id, dream_text) tuples
    Returns: (batch_ids, result_text, input_tokens, output_tokens)
    """
    batch_ids = [item[0] for item in batch_items]
    batch_texts = [item[1] for item in batch_items]
    
    # Build user prompt with batch data
    user_prompt_batch = user_prompt + "\n".join(batch_texts)
    
    async with semaphore:
        result, in_tokens, out_tokens = await call_llm(
            system_prompt, 
            user_prompt_batch, 
            session, 
            logger,
            timeout=180,
            model_used="deepseek-ai/DeepSeek-V3"
        )
    
    return batch_ids, result, in_tokens, out_tokens

In [9]:
# batch save results
from datetime import datetime

def save_results_batch(results, output_file):
    """Save successful results to JSONL file"""
    saved_count = 0
    for item in results:
        with open(output_file, "a", encoding="utf-8") as f:
            json_line = json.dumps(item, ensure_ascii=False)
            f.write(json_line + "\n")
        saved_count += 1
    return saved_count

def save_failed_batch(batch_ids, error_reason, failed_file):
    """Save failed batch info for later retry"""
    failed_entry = {
        "batch_ids": batch_ids,
        "error": error_reason,
        "timestamp": datetime.now().strftime('%Y%m%d_%H%M%S')
    }
    with open(failed_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(failed_entry, ensure_ascii=False) + "\n")

In [ ]:
# Main Async Processing Loop
async def main():
    logger.info("Starting dream processing...")
    
    if not dreams_to_process:
        logger.info("No new dreams to process. Exiting.")
        return
    
    # Create batches
    batches = []
    for i in range(0, len(dreams_to_process), BATCH_SIZE):
        batches.append(dreams_to_process[i:i+BATCH_SIZE])
    
    logger.info(f"Total batches to process: {len(batches)}")
    
    # Create semaphore for concurrency control
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
    
    total_saved = 0
    total_failed = 0
    total_input_tokens = 0
    total_output_tokens = 0
    
    async with aiohttp.ClientSession(
        connector=aiohttp.TCPConnector(limit=MAX_CONCURRENCY),
        timeout=aiohttp.ClientTimeout(total=600)
    ) as session:
        
        # Create all tasks
        tasks = [process_batch(batch, session, semaphore) for batch in batches]
        
        results_buffer = []
        
        # Process with progress bar
        for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Processing batches"):
            try:
                batch_ids, result, in_tokens, out_tokens = await task
                total_input_tokens += in_tokens
                total_output_tokens += out_tokens
                
                # Check for error or empty result
                if is_error_result(result):
                    logger.warning(f"Batch {batch_ids} failed with error: {result[:100] if result else 'empty'}")
                    save_failed_batch(batch_ids, result[:100] if result else "empty result", FAILED_FILE)
                    total_failed += len(batch_ids)
                    continue
                
                # Extract JSON from response (handles markdown wrapped responses)
                json_str = extract_json_from_response(result.strip())
                
                # Parse JSON result
                try:
                    result_list = json.loads(json_str)
                except json.JSONDecodeError as e:
                    logger.error(f"JSON parse error for batch {batch_ids}: {e}")
                    save_failed_batch(batch_ids, f"JSON parse error: {str(e)}", FAILED_FILE)
                    total_failed += len(batch_ids)
                    continue
                
                # Validate result
                is_valid, msg = validate_result(result_list, batch_ids)
                if not is_valid:
                    logger.warning(f"Validation failed for batch {batch_ids}: {msg}")
                    save_failed_batch(batch_ids, msg, FAILED_FILE)
                    total_failed += len(batch_ids)
                    continue
                
                # Add to buffer
                results_buffer.extend(result_list)
                
                # Save at intervals
                if len(results_buffer) >= SAVE_INTERVAL * BATCH_SIZE:
                    saved = save_results_batch(results_buffer, OUTPUT_FILE)
                    total_saved += saved
                    logger.info(f"Saved batch: {saved} dreams. Total saved: {total_saved}")
                    results_buffer = []
                    
            except Exception as e:
                logger.error(f"Critical error in processing loop: {e}")
        
        # Save remaining results
        if results_buffer:
            saved = save_results_batch(results_buffer, OUTPUT_FILE)
            total_saved += saved
            logger.info(f"Saved final batch: {saved} dreams")
    
    # Final statistics
    logger.info("="*50)
    logger.info("Processing complete!")
    logger.info(f"Total saved: {total_saved}")
    logger.info(f"Total failed: {total_failed}")
    logger.info(f"Total input tokens: {total_input_tokens}")
    logger.info(f"Total output tokens: {total_output_tokens}")
    if len(dreams_to_process) > 0:
        logger.info(f"Success rate: {total_saved/len(dreams_to_process)*100:.1f}%")

In [ ]:
await main()

2026-05-19 00:53:41,861 - INFO - Starting dream processing...
2026-05-19 00:53:41,864 - INFO - Total batches to process: 8425
Processing batches:  41%|████      | 3455/8425 [5:49:03<20:21:50, 14.75s/it]2026-05-19 06:42:50,346 - ERROR - Exception, attempt 1: [Errno 60] Operation timed out
2026-05-19 06:42:58,117 - ERROR - Exception, attempt 1: [Errno 60] Operation timed out
2026-05-19 06:43:08,377 - ERROR - Exception, attempt 1: [Errno 60] Operation timed out
2026-05-19 06:43:18,175 - ERROR - Exception, attempt 2: Cannot connect to host api.siliconflow.com:443 ssl:default [nodename nor servname provided, or not known]
2026-05-19 06:43:18,177 - ERROR - Exception, attempt 2: Cannot connect to host api.siliconflow.com:443 ssl:default [nodename nor servname provided, or not known]
2026-05-19 06:43:18,178 - ERROR - Exception, attempt 2: Cannot connect to host api.siliconflow.com:443 ssl:default [nodename nor servname provided, or not known]
2026-05-19 06:43:18,178 - ERROR - Exception, attemp

CancelledError: 

2026-05-19 11:46:59,378 - ERROR - Exception, attempt 3: Session is closed
2026-05-19 11:46:59,379 - ERROR - Failed due to exception: Session is closed
2026-05-19 11:46:59,380 - ERROR - Exception, attempt 1: Session is closed
2026-05-19 11:46:59,671 - ERROR - Exception, attempt 2: Session is closed
2026-05-19 11:46:59,774 - ERROR - Exception, attempt 3: Session is closed
2026-05-19 11:46:59,774 - ERROR - Failed due to exception: Session is closed
2026-05-19 11:46:59,776 - ERROR - Exception, attempt 1: Session is closed
2026-05-19 11:47:00,043 - ERROR - Exception, attempt 2: Session is closed
2026-05-19 11:47:00,292 - ERROR - Exception, attempt 2: Session is closed
2026-05-19 11:47:00,786 - ERROR - Exception, attempt 3: Session is closed
2026-05-19 11:47:00,788 - ERROR - Failed due to exception: Session is closed
2026-05-19 11:47:00,789 - ERROR - Exception, attempt 1: Session is closed
2026-05-19 11:47:00,934 - ERROR - Exception, attempt 2: Session is closed
2026-05-19 11:47:01,240 - ERR

## 时间估算

根据测试结果：
- **处理10个梦境**：169.79秒（约2分50秒）
- **平均处理1个梦境**：16.98秒

**完整处理45221个梦境的预估时间：**
- 总批次数：9045批次（每批次5个梦境）
- 并发数：10个并发请求
- 预估总时间：约 **25-30小时**
  - 理论计算：9045批次 × 85秒/批次 ÷ 10并发 ≈ 2.55小时
  - 实际考虑网络波动、rate limit、错误重试等，预计需要更长时间

###

### 实际的结果
钱烧的比我想象的快...  
我们充了10美刀，运行了大概12h，最终得到了27405条数据。  
数据量基本够用，所以我们也就没有继续跑了。  